# Day3_20 · 가까운 지역의 값 비교

전국 시·군·구 중심점과 공개 기온 격자를 이용해 가까운 지역끼리 값이 비슷한지 확인합니다.

## 이 노트북에서 확인할 내용

- 한 지역의 가까운 이웃을 표에서 확인할 수 있습니다.
- 지역값과 이웃 평균값을 연결할 수 있습니다.
- Moran's I와 무작위 배치 결과를 비교할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-20-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-20-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-20-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-20-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-20-05 · pandas와 numpy 불러오기

In [ ]:
import pandas as pd
import numpy as np

### D3B-20-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt

### D3B-20-07 · JSON 도구 불러오기

In [ ]:
import json

### D3B-20-08 · 전국 시·군·구 기온표 읽기

In [ ]:
spatial_weather = pd.read_csv(DATA_DIR / 'korea_adm2_temperature_climatology.csv')

In [ ]:
spatial_weather.head()

### D3B-20-09 · 시·군·구 행 수 확인

In [ ]:
spatial_weather.shape

### D3B-20-10 · 기온 범위 확인

In [ ]:
spatial_weather['temperature_mean_c'].describe().round(2)

### D3B-20-11 · 전국 시·군·구 경계 읽기

In [ ]:
korea_geo = json.loads((DATA_DIR / 'korea_adm2_simplified.geojson').read_text(encoding='utf-8'))

In [ ]:
pd.Series({'features': len(korea_geo['features']), 'type': korea_geo['type']})

### D3B-20-12 · 첫 지역의 이름 확인

In [ ]:
korea_geo['features'][0]['properties']

각 시·군·구의 대표 중심점에 NASA POWER 0.5° 기후 평년 격자 중 가장 가까운 연평균기온을 연결했습니다. 행정구역 전체의 관측소 평균과는 구분해야 합니다.

### D3B-20-13 · 경계 좌표를 꺼내는 함수 만들기

In [ ]:
def geometry_rings(geometry):
    if geometry['type'] == 'Polygon':
        return [geometry['coordinates'][0]]
    if geometry['type'] == 'MultiPolygon':
        return [polygon[0] for polygon in geometry['coordinates']]
    return []

### D3B-20-14 · 지역별 기온 찾기표 만들기

In [ ]:
temperature_by_id = spatial_weather.set_index('shape_id')['temperature_mean_c'].to_dict()

In [ ]:
pd.Series(temperature_by_id).head()

### D3B-20-15 · 전국 시·군·구 기온 지도 그리기

In [ ]:
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
fig, ax = plt.subplots(figsize=(7, 9))
norm = Normalize(spatial_weather['temperature_mean_c'].min(), spatial_weather['temperature_mean_c'].max())
for feature in korea_geo['features']:
    value = temperature_by_id[feature['properties']['shapeID']]
    color = plt.cm.coolwarm(norm(value))
    for ring in geometry_rings(feature['geometry']):
        x, y = zip(*ring)
        ax.fill(x, y, facecolor=color, edgecolor='white', linewidth=0.2)
ax.set_aspect('equal')
ax.set_axis_off()
ax.set_title('NASA POWER annual temperature climatology')
fig.colorbar(ScalarMappable(norm=norm, cmap='coolwarm'), ax=ax, label='Mean temperature (°C)', shrink=0.65)
plt.show()

## 가까운 지역 네 곳 찾기

행정경계 공유 여부 대신 각 대표점에서 가까운 중심점 네 곳을 이웃으로 정합니다. 이웃을 정하는 규칙이 바뀌면 결과도 달라질 수 있습니다.

### D3B-20-16 · 가까운 점 찾기 도구 불러오기

In [ ]:
from sklearn.neighbors import NearestNeighbors

### D3B-20-17 · 대표점 좌표표 만들기

In [ ]:
district_coordinates = spatial_weather[['longitude', 'latitude']]

In [ ]:
district_coordinates.head()

### D3B-20-18 · 자기 자신과 이웃 네 곳 찾는 규칙 만들기

In [ ]:
neighbor_model = NearestNeighbors(n_neighbors=5).fit(district_coordinates)

In [ ]:
neighbor_model

### D3B-20-19 · 모든 지역의 가까운 점 계산

In [ ]:
neighbor_distances, neighbor_positions = neighbor_model.kneighbors(district_coordinates)

In [ ]:
pd.DataFrame(neighbor_positions[:5])

### D3B-20-20 · 자기 자신을 제외한 이웃 위치 만들기

In [ ]:
neighbor_positions = neighbor_positions[:, 1:]

In [ ]:
pd.DataFrame(neighbor_positions[:5], columns=['neighbor_1', 'neighbor_2', 'neighbor_3', 'neighbor_4'])

### D3B-20-21 · 확인할 지역 이름 정하기

In [ ]:
# 아래 이름을 spatial_weather의 district_name 값으로 바꾸어 보세요.
selected_district = 'Ulju-gun'

In [ ]:
selected_district

### D3B-20-22 · 선택한 지역의 행 위치 찾기

In [ ]:
selected_position = spatial_weather.index[spatial_weather['district_name'] == selected_district][0]

In [ ]:
selected_position

### D3B-20-23 · 선택한 지역의 이웃 네 곳 확인

In [ ]:
selected_neighbors = spatial_weather.iloc[neighbor_positions[selected_position]][['district_name', 'longitude', 'latitude', 'temperature_mean_c']]

In [ ]:
selected_neighbors.round(2)

## 한 지역의 값과 이웃 평균을 연결하기

### D3B-20-24 · 기온 배열 만들기

In [ ]:
temperature_values = spatial_weather['temperature_mean_c'].to_numpy()

In [ ]:
pd.Series(temperature_values[:10])

### D3B-20-25 · 각 지역의 이웃 평균기온 계산

In [ ]:
neighbor_temperature_mean = temperature_values[neighbor_positions].mean(axis=1)

In [ ]:
pd.Series(neighbor_temperature_mean[:10], name='neighbor_mean_temperature_c').round(2)

### D3B-20-26 · 지역값과 이웃 평균을 한 표로 만들기

In [ ]:
spatial_table = spatial_weather.copy()
spatial_table['neighbor_mean_temperature_c'] = neighbor_temperature_mean

In [ ]:
spatial_table[['district_name', 'temperature_mean_c', 'neighbor_mean_temperature_c']].head().round(2)

### D3B-20-27 · 지역기온과 이웃 평균기온 산점도

In [ ]:
plt.scatter(spatial_table['temperature_mean_c'], spatial_table['neighbor_mean_temperature_c'], alpha=0.6)
plt.xlabel('District temperature (°C)')
plt.ylabel('Mean temperature of 4 nearest districts (°C)')
plt.show()

### D3B-20-28 · 지역기온과 이웃 평균의 상관계수

In [ ]:
spatial_neighbor_correlation = spatial_table['temperature_mean_c'].corr(spatial_table['neighbor_mean_temperature_c'])

In [ ]:
spatial_neighbor_correlation

가까운 지역끼리 비슷한 값을 갖는 정도를 하나의 수치로 요약합니다. 전문가들은 이를 공간적 자기상관이라고 부릅니다.

### D3B-20-29 · 이웃 관계표 만들기

In [ ]:
number_of_districts = len(spatial_weather)
weight_matrix = np.zeros((number_of_districts, number_of_districts))
for row_position, neighbors in enumerate(neighbor_positions):
    weight_matrix[row_position, neighbors] = 1

In [ ]:
pd.DataFrame(weight_matrix[:8, :8])

### D3B-20-30 · Moran's I 계산 함수 만들기

In [ ]:
def moran_i(values, weights):
    centered = values - values.mean()
    numerator = (weights * np.outer(centered, centered)).sum()
    denominator = (centered ** 2).sum()
    return len(values) / weights.sum() * numerator / denominator

### D3B-20-31 · 관찰된 Moran's I 계산

In [ ]:
observed_moran = moran_i(temperature_values, weight_matrix)

In [ ]:
observed_moran

양의 Moran's I는 비슷한 값이 가까이 모이는 경향을, 음의 값은 서로 다른 값이 번갈아 나타나는 경향을 뜻합니다. 값의 크기는 이웃 규칙과 공간 단위에 의존합니다.

### D3B-20-32 · 무작위 배치 499개 만들기

In [ ]:
rng = np.random.default_rng(42)
permuted_moran = pd.Series([moran_i(rng.permutation(temperature_values), weight_matrix) for _ in range(499)], name='Moran I under random placement')

In [ ]:
permuted_moran.head()

### D3B-20-33 · 양의 군집에 대한 순열 p-value 계산

In [ ]:
spatial_p_value = ((permuted_moran >= observed_moran).sum() + 1) / (len(permuted_moran) + 1)

In [ ]:
spatial_p_value

### D3B-20-34 · 무작위 값과 관찰값 비교

In [ ]:
plt.hist(permuted_moran, bins=30, color='lightgray', edgecolor='white')
plt.axvline(observed_moran, color='red', linewidth=2, label='Observed')
plt.xlabel("Moran's I")
plt.legend()
plt.show()

### D3B-20-35 · 공간 결과표 만들기

In [ ]:
spatial_result = pd.Series({
    'district_rows': len(spatial_weather),
    'neighbors_per_district': neighbor_positions.shape[1],
    'observed_Moran_I': observed_moran,
    'right_tail_permutation_p': spatial_p_value,
})

In [ ]:
spatial_result.round(4)

현재 결과는 시·군·구 중심점과 가장 가까운 NASA 격자값을 사용한 실습입니다. 격자 기온 자체가 공간적으로 부드럽게 만들어진 자료이므로, 행정구역 정책이나 지역 원인을 증명한 결과가 아닙니다.

### D3B-20-36 · 바꾸어 볼 이웃 수 정하기

In [ ]:
# 4를 6 또는 8로 바꾸어 보세요.
chosen_neighbor_count = 6

In [ ]:
chosen_neighbor_count

### D3B-20-37 · 새 이웃 수로 위치 계산

In [ ]:
chosen_neighbor_model = NearestNeighbors(n_neighbors=chosen_neighbor_count + 1).fit(district_coordinates)
chosen_positions = chosen_neighbor_model.kneighbors(district_coordinates, return_distance=False)[:, 1:]

In [ ]:
pd.DataFrame(chosen_positions[:5])

### D3B-20-38 · 새 이웃 관계표 만들기

In [ ]:
chosen_weights = np.zeros((number_of_districts, number_of_districts))
for row_position, neighbors in enumerate(chosen_positions):
    chosen_weights[row_position, neighbors] = 1

In [ ]:
pd.Series(chosen_weights.sum(axis=1)).value_counts().sort_index()

### D3B-20-39 · 새 이웃 수의 Moran's I 계산

In [ ]:
chosen_moran = moran_i(temperature_values, chosen_weights)

In [ ]:
chosen_moran

### D3B-20-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''spatial_weather, neighbor_positions, temperature_values를 사용하세요. 기존 셀은 수정하지 마세요. 선택한 지역과 가까운 네 지역을 지도에 다른 색으로 표시하는 새 셀을 제안해 주세요. 중심점 기반 이웃과 행정경계 공유 이웃이 다를 수 있다는 설명을 포함해 주세요.'''.strip()

In [ ]:
codex_request

### D3B-20-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['선택한 지역과 이웃을 서로 다른 색으로 표시하는가?', '경도와 위도 순서가 맞는가?', '가까운 중심점과 경계 공유를 같은 것으로 말하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist